## Linearization Pipeline

This notebook has code that will run the entire linearization process on a single data file.

#### Instructions

1. Edit the assignment statements in the code cell below to define paths to data files and other options (eventually these will be read from the command line or a configuration file).

2. Execute all the code cells.  You can click "Run All" to execute every step, then scroll to the bottom to view the plot showing the linearized data set.  Alternatively, step through the code once cell at a time.  At various places where will be code cells that have been commented out; uncomment them if you want to see a more detailed output.

In [1]:
# Define the path to the project directory.  It can be an absolute path or a path relative to this notebook.

from pathlib import Path

TEST_DATA = '210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx'

PROJECT_DIR = Path.home() / 'Research/Projects/LibudaLab/GAP/PRG-1'
IMARIS_FILE = PROJECT_DIR / 'xlsx' / TEST_DATA
MEIOTIC_STAGES = PROJECT_DIR / 'Germline Measurements.xlsx'

#### Notebook Organization

Each step in the analysis pipeline is in a separate section that starts with a level 3 headers ("Read Position Data", "Compute Perpendicular Intersections", _etc_.)

Most sections define one or more Python functions and then call those functions.  The idea is to simplify the notebook's global namespace.
* objects that only used temporarily as part of a single step are saved in local variables of the functions
* any data that will be used in later steps is saved in a global variable

Local variables typically have short names that should be understandable in context, _e.g._ `mf` stands for "measurement frame" in the section that reads the line segment locations.

Global variables will have longer names that should be recognizable later, _e.g._ `segments` will be a frame that has all the information about line segments.  To make it easier to find cells where the variables are defined comments before the assignment statement are marked with ★

### Imports

In [2]:
# Installed data science libraries 
import geopandas as gp
import numpy as np
import pandas as pd
from shapely.geometry import Point, LineString

# Installed graphics libraries
from bokeh.palettes import Sunset10
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.transform import linear_cmap


### Read Position Data

Define a function that gets $x$ and $y$ coordinates from the Position sheet of the Imaris data file.  The function will be called twice, once to read measurement (ends of line segments) locations and once to read locations of nuclei.

In [3]:
def read_positions(f: pd.ExcelFile, category: str):
    '''
    Read the "Position" sheet from an XLS file exported by Imaris.  Select data points
    in the specified category, return them in a data frame that has the coordinates and
    IDs of the selected data.  Raises an exception if the file does not have a sheet 
    named "Position".

    Arguments:
        f:  the file to read from
        category: the type of data to read (based on the Category column in the sheet)

    Returns:
        a Pandas frame with names and Point objects
    '''
    assert "Position" in f.sheet_names, "spreadheet is missing the 'Position' sheet"

    sheet = f.parse("Position", header=1)

    pf = sheet[sheet['Category']==category]
    pf.index = range(len(pf))

    if category == 'MeasurementPoint':
        id_col = pd.Series(pf['Name'])
        id_name = 'name'
    else:
        id_col = pd.Series((pf['Surpass Object'] + pf['ID'].apply(str)), name='ID')
        id_name = 'id'

    df = gp.GeoDataFrame({
        id_name: id_col,
        'point': [Point(pf.loc[i]['Position X'], pf.loc[i]['Position Y']) for i in range(len(pf))]
    }).set_geometry('point')

    return df


★ Read the end points of the line segments and the locations of nuclei, save them in global variables for future steps.

In [4]:
with pd.ExcelFile(IMARIS_FILE, engine='calamine') as f:
    measurements = read_positions(f, category='MeasurementPoint')
    nuclei = read_positions(f, category='Surface')

**Optional:**  Uncomment this code cell to see the first few measurements.

In [5]:
measurements.head()

,name,point
0,A,POINT (38.501 141.542)
1,B,POINT (39.457 120.56)
2,C,POINT (40.56 110.826)
3,D,POINT (43.907 99.405)
4,E,POINT (48.822 81.404)


**Optional:**  Uncomment this code cell to see the first few nuclei.

In [6]:
nuclei.head()

,id,point
0,prg1_dk0,POINT (21.794 138.72)
1,prg1_dk1,POINT (19.615 141.887)
2,prg1_dk2,POINT (26.146 135.15)
3,prg1_dk3,POINT (21.147 141.275)
4,prg1_dk4,POINT (25.013 137.296)


In [7]:
nuclei.tail()

,id,point
4574,prg1_tz1215,POINT (53.12 8.437)
4575,prg1_tz1216,POINT (53.845 8.47)
4576,prg1_tz1217,POINT (52.603 8.754)
4577,prg1_tz1218,POINT (54.63 6.596)
4578,prg1_tz1219,POINT (51.939 9.409)


### Read Meitotic Stage Data

In [8]:
def read_stages(fn):
    xls_file_name = fn.replace('.xlsx','.xls')
    with pd.ExcelFile(MEIOTIC_STAGES, engine='calamine') as f:
        df = f.parse().set_index('GonadID')
        row = df.loc[xls_file_name]
    return {s: row[s] for s in ['TZ_start','TZ_end','Pachy_end']}

**Optional:**  Uncomment this code cell to see the meiotic stages for the data set

In [9]:
read_stages(TEST_DATA)

{'TZ_start': 'G', 'TZ_end': 'L', 'Pachy_end': 'Q'}

★ Read the meiotic stages, save them in global variables for future steps.

In [10]:
stages = read_stages(TEST_DATA)

### Create Line Segments

Define a function that creates line segments from adjacent measurements.

In [48]:
def create_segments(mf, sd=None):
    '''
    Make a frame where rows contain descriptions of line segments made by connecting
    adjacent points in the measurement frame.  Adds columns with attributes of each
    segment, including the parameters of the linear equation and segment length.

    Arguments:
        mf:  a data frame with names and locations of measurement points
        sd:  (optional) a dictionary with meiotic stage definitions

    Returns:
        a Pandas frame with line segments and their equations and lengths.
    '''
    df = gp.GeoDataFrame({
        'name': [mf.loc[i]['name'] + mf.loc[i+1]['name'] for i in range(len(mf)-1)],
        'head': [mf['point'].loc[i] for i in range(len(mf)-1)],
        'tail': [mf['point'].loc[i+1] for i in range(len(mf)-1)],
    }).set_geometry('head').set_geometry('tail')

    df['segment'] = [LineString([df['head'][i],df['tail'][i]]) for i in range(len(df))]
    df['A'] = ((df['tail'].y - df['head'].y) / (df['tail'].x - df['head'].x))
    df['C'] = df['head'].y - df['head'].x*df['A']
    df['length'] = Point.distance(df['head'], df['tail'])
    df['pathlen'] = np.cumulative_sum(df['length'], include_initial=True)[:-1]

    if sd:
        df['stage'] = [('PMT' if p < sd['TZ_start'] else 'TZ' if p < sd['TZ_end'] else 'PACH') for p in mf['name'][0:-1]]
    else:
        df['stage'] = ['n/a'] * (len(mf)-1)
   
    return df.set_geometry('segment')

★ Save the line segments in a global variable for future steps.  Uncomment one of these lines, depending on whether or not the data set has meiotic stages.

In [49]:
# segments = create_segments(measurements)
segments = create_segments(measurements, stages)

In [50]:
segments

,name,head,tail,segment,A,C,length,pathlen,stage
0,AB,POINT (38.501 141.542),POINT (39.457 120.56),"LINESTRING (38.501 141.542, 39.457 120.56)",-21.947680,986.549610,21.003778,0.000000,PMT
1,BC,POINT (39.457 120.56),POINT (40.56 110.826),"LINESTRING (39.457 120.56, 40.56 110.826)",-8.825019,468.768762,9.796295,21.003778,PMT
2,CD,POINT (40.56 110.826),POINT (43.907 99.405),"LINESTRING (40.56 110.826, 43.907 99.405)",-3.412309,249.229242,11.901328,30.800072,PMT
3,DE,POINT (43.907 99.405),POINT (48.822 81.404),"LINESTRING (43.907 99.405, 48.822 81.404)",-3.662464,260.212807,18.659935,42.701400,PMT
4,EF,POINT (48.822 81.404),POINT (54.427 66.404),"LINESTRING (48.822 81.404, 54.427 66.404)",-2.676182,212.060563,16.012995,61.361335,PMT
5,FG,POINT (54.427 66.404),POINT (61.472 53.318),"LINESTRING (54.427 66.404, 61.472 53.318)",-1.857487,167.501433,14.861878,77.374330,PMT
6,GH,POINT (61.472 53.318),POINT (67.041 44.591),"LINESTRING (61.472 53.318, 67.041 44.591)",-1.567068,149.648795,10.352503,92.236208,TZ
7,HI,POINT (67.041 44.591),POINT (72.587 25.798),"LINESTRING (67.041 44.591, 72.587 25.798)",-3.388570,271.764099,19.594257,102.588711,TZ
8,IJ,POINT (72.587 25.798),POINT (69.01 14.931),"LINESTRING (72.587 25.798, 69.01 14.931)",3.038024,-194.723069,11.440568,122.182968,TZ
9,JK,POINT (69.01 14.931),POINT (61.052 11.458),"LINESTRING (69.01 14.931, 61.052 11.458)",0.436416,-15.186063,8.682831,133.623536,TZ


★ Save the total length of all line segments in a global variable

In [13]:
germline_length = np.sum(segments['length'])

**Optional:**  Uncomment this code cell to see the first few line segments.

In [14]:
segments.head()

,name,head,tail,segment,A,C,length,pathlen
0,AB,POINT (38.501 141.542),POINT (39.457 120.56),"LINESTRING (38.501 141.542, 39.457 120.56)",-21.947680,986.549610,21.003778,0.000000
1,BC,POINT (39.457 120.56),POINT (40.56 110.826),"LINESTRING (39.457 120.56, 40.56 110.826)",-8.825019,468.768762,9.796295,21.003778
2,CD,POINT (40.56 110.826),POINT (43.907 99.405),"LINESTRING (40.56 110.826, 43.907 99.405)",-3.412309,249.229242,11.901328,30.800072
3,DE,POINT (43.907 99.405),POINT (48.822 81.404),"LINESTRING (43.907 99.405, 48.822 81.404)",-3.662464,260.212807,18.659935,42.701400
4,EF,POINT (48.822 81.404),POINT (54.427 66.404),"LINESTRING (48.822 81.404, 54.427 66.404)",-2.676182,212.060563,16.012995,61.361335


In [15]:
segments.tail()

,name,head,tail,segment,A,C,length,pathlen
11,LM,POINT (50.432 11.52),POINT (45.424 19.481),"LINESTRING (50.432 11.52, 45.424 19.481)",-1.589657,91.689572,9.405189,152.926547
12,MN,POINT (45.424 19.481),POINT (41.883 26.756),"LINESTRING (45.424 19.481, 41.883 26.756)",-2.054504,112.804793,8.091001,162.331736
13,NO,POINT (41.883 26.756),POINT (36.718 37.509),"LINESTRING (41.883 26.756, 36.718 37.509)",-2.081897,113.952079,11.929133,170.422738
14,OP,POINT (36.718 37.509),POINT (23.646 69.875),"LINESTRING (36.718 37.509, 23.646 69.875)",-2.475980,128.422012,34.906091,182.351870
15,PQ,POINT (23.646 69.875),POINT (17.912 91.123),"LINESTRING (23.646 69.875, 17.912 91.123)",-3.705616,157.498003,22.008096,217.257961


**Optional:**  Uncomment the code in this cell to see a drawing of the line segments (using location data in the measurements frame)

In [16]:
output_notebook()

xs = list(measurements['point'].x)
ys = list(measurements['point'].y)
ts = list(measurements['name'])
p = figure(title='demo', x_axis_label='x', y_axis_label='y', match_aspect=True)
p.line(x=xs, y=ys)
p.scatter(x=xs, y=ys, size=5)
p.text(x=xs, y=ys, text=ts, x_offset=5, y_offset=10)
show(p)

Loading BokehJS ...

### Make a Frame with All Combinations of Nuclei and Line Segments

Prepare for the step that computes distances from nuclei to segments by making a frame that has every combination of nuclei and segment descriptions.

★ Save the combined data in a global variable for future steps.

In [17]:
combined = gp.GeoDataFrame.join(nuclei, segments, how="cross")

It's a lot of rows...

In [18]:
len(combined)

73264

Verify the combined frame has the expected number of rows.

In [19]:
assert len(combined) == len(nuclei) * len(segments)

**Optional:**  Uncomment this code cell to see the first few lines in the combined frame.

In [20]:
combined.head()

,id,point,name,head,tail,segment,A,C,length,pathlen
0,prg1_dk0,POINT (21.794 138.72),AB,POINT (38.501 141.542),POINT (39.457 120.56),"LINESTRING (38.501 141.542, 39.457 120.56)",-21.947680,986.549610,21.003778,0.000000
1,prg1_dk0,POINT (21.794 138.72),BC,POINT (39.457 120.56),POINT (40.56 110.826),"LINESTRING (39.457 120.56, 40.56 110.826)",-8.825019,468.768762,9.796295,21.003778
2,prg1_dk0,POINT (21.794 138.72),CD,POINT (40.56 110.826),POINT (43.907 99.405),"LINESTRING (40.56 110.826, 43.907 99.405)",-3.412309,249.229242,11.901328,30.800072
3,prg1_dk0,POINT (21.794 138.72),DE,POINT (43.907 99.405),POINT (48.822 81.404),"LINESTRING (43.907 99.405, 48.822 81.404)",-3.662464,260.212807,18.659935,42.701400
4,prg1_dk0,POINT (21.794 138.72),EF,POINT (48.822 81.404),POINT (54.427 66.404),"LINESTRING (48.822 81.404, 54.427 66.404)",-2.676182,212.060563,16.012995,61.361335


In [21]:
combined.sample(n=10)

,id,point,name,head,tail,segment,A,C,length,pathlen
50747,prg1_pm528,POINT (46.417 72.094),LM,POINT (50.432 11.52),POINT (45.424 19.481),"LINESTRING (50.432 11.52, 45.424 19.481)",-1.589657,91.689572,9.405189,152.926547
66603,prg1_tz803,POINT (73.032 11.217),LM,POINT (50.432 11.52),POINT (45.424 19.481),"LINESTRING (50.432 11.52, 45.424 19.481)",-1.589657,91.689572,9.405189,152.926547
34510,prg1_mp31,POINT (36.709 61.585),OP,POINT (36.718 37.509),POINT (23.646 69.875),"LINESTRING (36.718 37.509, 23.646 69.875)",-2.475980,128.422012,34.906091,182.351870
54638,prg1_tz55,POINT (72.843 45.922),OP,POINT (36.718 37.509),POINT (23.646 69.875),"LINESTRING (36.718 37.509, 23.646 69.875)",-2.475980,128.422012,34.906091,182.351870
16679,prg1_ep420,POINT (39.408 20.839),HI,POINT (67.041 44.591),POINT (72.587 25.798),"LINESTRING (67.041 44.591, 72.587 25.798)",-3.388570,271.764099,19.594257,102.588711
5565,prg1_dk347,POINT (10.438 117.13),NO,POINT (41.883 26.756),POINT (36.718 37.509),"LINESTRING (41.883 26.756, 36.718 37.509)",-2.081897,113.952079,11.929133,170.422738
30846,prg1_lp601,POINT (12.441 77.356),OP,POINT (36.718 37.509),POINT (23.646 69.875),"LINESTRING (36.718 37.509, 23.646 69.875)",-2.475980,128.422012,34.906091,182.351870
6609,prg1_dk413,POINT (8.494 115.912),BC,POINT (39.457 120.56),POINT (40.56 110.826),"LINESTRING (39.457 120.56, 40.56 110.826)",-8.825019,468.768762,9.796295,21.003778
1291,prg1_dk80,POINT (25.278 124.809),LM,POINT (50.432 11.52),POINT (45.424 19.481),"LINESTRING (50.432 11.52, 45.424 19.481)",-1.589657,91.689572,9.405189,152.926547
5053,prg1_dk315,POINT (21.484 109.204),NO,POINT (41.883 26.756),POINT (36.718 37.509),"LINESTRING (41.883 26.756, 36.718 37.509)",-2.081897,113.952079,11.929133,170.422738


In [22]:
type(combined)

geopandas.geodataframe.GeoDataFrame

In [23]:
combined.dtypes

id              str
point      geometry
name            str
head       geometry
tail       geometry
segment    geometry
A           float64
C           float64
length      float64
pathlen     float64
dtype: object

### Compute Distances

Define a function that computes three distances for each nucleus and line segment: the distance from a the nucleus to each end point (`pa` and `pb`), and the distance to the line segment (`pm`) 

In [24]:
def add_distances(df: gp.GeoDataFrame):
    '''
    Create a new frame that summarizes the distances between nuclei and line segments.  The
    columns in the new frame will be:
    * `nuc_id`, the nucleus ID
    * `seg_name`, the name of the segment
    * `distance`, the distance from the nucleus to the segment
    * `location`, a string that identifies where the nucleus is closest (head, tail, middle of the segment)
    
    Arguments:
      df: a GeoDataFrame that has the line segments and their equations

    Returns:
      a frame with the distance values
    '''
    
    pos = gp.GeoDataFrame({
        'head': gp.GeoDataFrame.distance(df['point'],df['head']),
        'tail': gp.GeoDataFrame.distance(df['point'],df['tail']),
        'mid': gp.GeoDataFrame.distance(df['point'],df['segment']),
    })

    res = gp.GeoDataFrame({
        'nuc_id': df['id'],
        'seg_name': df['name'],
        'distance': pos.min(axis='columns'),
        'intersection': pos.idxmin(axis='columns')
    })

    return res

In [25]:
distf = add_distances(combined)

In [26]:
distf

,nuc_id,seg_name,distance,intersection
0,prg1_dk0,AB,16.818130,mid
1,prg1_dk0,BC,25.333127,head
2,prg1_dk0,CD,33.619017,head
3,prg1_dk0,DE,45.107142,head
4,prg1_dk0,EF,63.369052,head
...,...,...,...,...
73259,prg1_tz1219,LM,2.593717,head
73260,prg1_tz1219,MN,11.995433,head
73261,prg1_tz1219,NO,20.050974,head
73262,prg1_tz1219,OP,31.957609,head


### Groups

Group the distances by nucleus, and then find the shortest distance within each group.  The 
result of executing this expression is a column of row numbers, where each row number is the
row in the original frame that has the shortest distance to a nucleus.

In [27]:
distf.groupby('nuc_id')[['distance']].min()

,distance
nuc_id,
prg1_dk0,16.818130
prg1_dk1,18.889150
prg1_dk10,13.355502
prg1_dk100,22.382311
prg1_dk101,21.825607
...,...
prg1_tz995,5.361531
prg1_tz996,4.307980
prg1_tz997,2.744620


Use those column numbers to select the rows in the distance frame to get a complete description of a nucleus and the closest line segment.

In [28]:
locs = distf.groupby('nuc_id')[['distance']].idxmin()
distf.loc[locs.distance]

,nuc_id,seg_name,distance,intersection
0,prg1_dk0,AB,16.818130,mid
16,prg1_dk1,AB,18.889150,head
160,prg1_dk10,AB,13.355502,mid
1600,prg1_dk100,AB,22.382311,mid
1616,prg1_dk101,AB,21.825607,mid
...,...,...,...,...
69673,prg1_tz995,JK,5.361531,mid
69689,prg1_tz996,JK,4.307980,mid
69705,prg1_tz997,JK,2.744620,mid
69721,prg1_tz998,JK,3.861500,mid


Add the columns from the original frame so we have the data to compute intersections.

Save the result in a frame named `merged`.

In [29]:
merged = gp.GeoDataFrame(distf.loc[locs.distance].join(combined[['point','head','tail','A','C','pathlen']]))

In [30]:
merged.head()

,nuc_id,seg_name,distance,intersection,point,head,tail,A,C,pathlen
0,prg1_dk0,AB,16.818130,mid,POINT (21.794 138.72),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0
16,prg1_dk1,AB,18.889150,head,POINT (19.615 141.887),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0
160,prg1_dk10,AB,13.355502,mid,POINT (25.225 139.493),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0
1600,prg1_dk100,AB,22.382311,mid,POINT (16.472 133.278),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0
1616,prg1_dk101,AB,21.825607,mid,POINT (17.102 131.682),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0


In [31]:
merged.dtypes

nuc_id               str
seg_name             str
distance         float64
intersection         str
point           geometry
head            geometry
tail            geometry
A                float64
C                float64
pathlen          float64
dtype: object

In [32]:
type(merged)

geopandas.geodataframe.GeoDataFrame

### Compute Perpendicular Intersections

Add an intersection point to each row in the merged data.
* if the intersection is the middle of a line segment we need to compute the perpendicular intersection point using the equations of the line
* otherwise use one of the end points as the intersection location

In [33]:
def add_intersections(df: gp.GeoDataFrame):
    '''
    Determine the location where nuclei intersect line segments.  

    Arguments:
      df: a frame that has line segments and their equations

    Returns:
      a copy of the frame with a new location column added
    '''
    mids = df[df['intersection']=='mid']
    xp = (mids.point.y - (-mids.point.x/mids.A) - mids.C) / (mids.A - (-1/mids.A))
    yp = (-xp/mids.A) + mids.point.y - (-mids.point.x/mids.A)
    mids['loc'] = [Point(xp.iloc[i],yp.iloc[i]) for i in range(len(mids))]

    heads = df[df['intersection']=='head']
    heads['loc'] = heads['head']

    tails = df[df['intersection']=='tail']
    tails['loc'] = tails['tail']

    res = pd.concat([mids,heads,tails]).set_geometry('loc')
    res['pathseg'] = gp.GeoDataFrame.distance(res['head'], res['loc'])
    res['pathloc'] = (res['pathlen'] + res['pathseg']) / germline_length
    
    return res

★ Save the intersections in a global variable for future steps.

In [34]:
intersections = add_intersections(merged)

In [35]:
intersections.head()

,nuc_id,seg_name,distance,intersection,point,head,tail,A,C,pathlen,loc,pathseg,pathloc
0,prg1_dk0,AB,16.818130,mid,POINT (21.794 138.72),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (38.595 139.485),2.058651,0.008604
160,prg1_dk10,AB,13.355502,mid,POINT (25.225 139.493),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (38.567 140.101),1.442622,0.006029
1600,prg1_dk100,AB,22.382311,mid,POINT (16.472 133.278),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (38.831 134.297),7.252778,0.030313
1616,prg1_dk101,AB,21.825607,mid,POINT (17.102 131.682),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (38.905 132.675),8.875792,0.037096
1632,prg1_dk102,AB,17.702667,mid,POINT (21.386 128.241),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (39.07 129.047),12.508224,0.052277


In [36]:
intersections.tail()

,nuc_id,seg_name,distance,intersection,point,head,tail,A,C,pathlen,loc,pathseg,pathloc
67592,prg1_tz865,IJ,6.420810,tail,POINT (71.696 9.099),POINT (72.587 25.798),POINT (69.01 14.931),3.038024,-194.723069,122.182968,POINT (69.01 14.931),11.440568,0.558473
67752,prg1_tz875,IJ,6.870928,tail,POINT (72.472 8.996),POINT (72.587 25.798),POINT (69.01 14.931),3.038024,-194.723069,122.182968,POINT (69.01 14.931),11.440568,0.558473
67832,prg1_tz880,IJ,7.073067,tail,POINT (73.041 9.119),POINT (72.587 25.798),POINT (69.01 14.931),3.038024,-194.723069,122.182968,POINT (69.01 14.931),11.440568,0.558473
53894,prg1_tz9,GH,8.016581,tail,POINT (74.522 47.472),POINT (61.472 53.318),POINT (67.041 44.591),-1.567068,149.648795,92.236208,POINT (67.041 44.591),10.352503,0.428764
55206,prg1_tz91,GH,3.040938,tail,POINT (69.791 45.889),POINT (61.472 53.318),POINT (67.041 44.591),-1.567068,149.648795,92.236208,POINT (67.041 44.591),10.352503,0.428764


### Assign a Meiotic Stage to Each Point

In [70]:
pmt = set(segments[segments['stage']=='PMT'].name)
pmf = intersections[intersections.seg_name.isin(pmt)]
pmf['stage'] = 'PMT'

In [71]:
tz = set(segments[segments['stage']=='TZ'].name)
tf = intersections[intersections.seg_name.isin(tz)]
tf['stage'] = 'TZ'

In [76]:
ps = set(segments[segments['stage']=='PACH'].name)
epf = intersections[intersections.seg_name.isin(ps) & intersections.pathloc > 0.9]

In [77]:
epf

,nuc_id,seg_name,distance,intersection,point,head,tail,A,C,pathlen,loc,pathseg,pathloc
7103,prg1_dk443,PQ,9.965052,mid,POINT (28.172 91.351),POINT (23.646 69.875),POINT (17.912 91.123),-3.705616,157.498003,217.257961,POINT (18.551 88.755),19.555074,0.989748
7167,prg1_dk447,PQ,10.725541,mid,POINT (29.025 91.109),POINT (23.646 69.875),POINT (17.912 91.123),-3.705616,157.498003,217.257961,POINT (18.67 88.315),19.099194,0.987842
7247,prg1_dk452,PQ,11.837555,mid,POINT (30.183 91.086),POINT (23.646 69.875),POINT (17.912 91.123),-3.705616,157.498003,217.257961,POINT (18.754 88.002),18.775280,0.986489
7263,prg1_dk453,PQ,11.730319,mid,POINT (29.971 91.46),POINT (23.646 69.875),POINT (17.912 91.123),-3.705616,157.498003,217.257961,POINT (18.646 88.404),19.191598,0.988229
8015,prg1_dk500,PQ,3.450678,mid,POINT (21.386 91.494),POINT (23.646 69.875),POINT (17.912 91.123),-3.705616,157.498003,217.257961,POINT (18.054 90.595),21.461168,0.997714
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33166,prg1_lp746,OP,9.386203,tail,POINT (14.586 67.422),POINT (36.718 37.509),POINT (23.646 69.875),-2.475980,128.422012,182.351870,POINT (23.646 69.875),34.906091,0.908018
33406,prg1_lp761,OP,8.365042,tail,POINT (15.728 67.177),POINT (36.718 37.509),POINT (23.646 69.875),-2.475980,128.422012,182.351870,POINT (23.646 69.875),34.906091,0.908018
33470,prg1_lp765,OP,9.665651,tail,POINT (14.53 66.662),POINT (36.718 37.509),POINT (23.646 69.875),-2.475980,128.422012,182.351870,POINT (23.646 69.875),34.906091,0.908018
33550,prg1_lp770,OP,9.328590,tail,POINT (14.984 66.412),POINT (36.718 37.509),POINT (23.646 69.875),-2.475980,128.422012,182.351870,POINT (23.646 69.875),34.906091,0.908018


In [73]:
ps

{'LM', 'MN', 'NO', 'OP', 'PQ'}

In [69]:
pd.concat([pf,tf])

,nuc_id,seg_name,distance,intersection,point,head,tail,A,C,pathlen,loc,pathseg,pathloc,stage
0,prg1_dk0,AB,16.818130,mid,POINT (21.794 138.72),POINT (38.501 141.542),POINT (39.457 120.56),-21.947680,986.549610,0.000000,POINT (38.595 139.485),2.058651,0.008604,PMT
160,prg1_dk10,AB,13.355502,mid,POINT (25.225 139.493),POINT (38.501 141.542),POINT (39.457 120.56),-21.947680,986.549610,0.000000,POINT (38.567 140.101),1.442622,0.006029,PMT
1600,prg1_dk100,AB,22.382311,mid,POINT (16.472 133.278),POINT (38.501 141.542),POINT (39.457 120.56),-21.947680,986.549610,0.000000,POINT (38.831 134.297),7.252778,0.030313,PMT
1616,prg1_dk101,AB,21.825607,mid,POINT (17.102 131.682),POINT (38.501 141.542),POINT (39.457 120.56),-21.947680,986.549610,0.000000,POINT (38.905 132.675),8.875792,0.037096,PMT
1632,prg1_dk102,AB,17.702667,mid,POINT (21.386 128.241),POINT (38.501 141.542),POINT (39.457 120.56),-21.947680,986.549610,0.000000,POINT (39.07 129.047),12.508224,0.052277,PMT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67592,prg1_tz865,IJ,6.420810,tail,POINT (71.696 9.099),POINT (72.587 25.798),POINT (69.01 14.931),3.038024,-194.723069,122.182968,POINT (69.01 14.931),11.440568,0.558473,TZ
67752,prg1_tz875,IJ,6.870928,tail,POINT (72.472 8.996),POINT (72.587 25.798),POINT (69.01 14.931),3.038024,-194.723069,122.182968,POINT (69.01 14.931),11.440568,0.558473,TZ
67832,prg1_tz880,IJ,7.073067,tail,POINT (73.041 9.119),POINT (72.587 25.798),POINT (69.01 14.931),3.038024,-194.723069,122.182968,POINT (69.01 14.931),11.440568,0.558473,TZ
53894,prg1_tz9,GH,8.016581,tail,POINT (74.522 47.472),POINT (61.472 53.318),POINT (67.041 44.591),-1.567068,149.648795,92.236208,POINT (67.041 44.591),10.352503,0.428764,TZ


In [ ]:
delta = (pf['pathloc'].max() - pf['pathloc'].min()) / 3

In [ ]:
intersections[(intersections.pathloc > pf['pathloc'].min()) & (intersections.pathloc <= pf['pathloc'].max())]

In [ ]:
a = pf['pathloc'].min()
for s in ['EP','MP','LP']:
    b = a + delta
    intersections[(intersections.pathloc > a) & (intersections.pathloc <= b)]['stage'].replace('PACH',s)
    a += delta

In [ ]:
intersections[intersections.stage == 'PACH']

In [ ]:
intersections[intersections.stage == 'EP']

In [94]:
def assign_stages(df):
    res = []
    for s in ['PMT','TZ']:
        sn = set(segments[segments['stage']==s].name)
        sf = intersections[intersections.seg_name.isin(sn)]
        sf['stage'] = s
        res.append(sf)
    sn = set(segments[segments['stage']=='PACH'].name)
    sf = intersections[intersections.seg_name.isin(sn)]
    delta = (sf['pathloc'].max() - sf['pathloc'].min()) / 3
    a = sf['pathloc'].min()
    for s in ['EP','MP','LP']:
        b = a + delta
        pf = intersections[(intersections.pathloc >= a) & (intersections.pathloc <= b)]
        pf['stage'] = s
        res.append(pf)
        a += delta
    return pd.concat(res)

In [95]:
final = assign_stages(intersections)

In [96]:
len(intersections), len(final)

(4579, 4579)

In [97]:
final.head()

,nuc_id,seg_name,distance,intersection,point,head,tail,A,C,pathlen,loc,pathseg,pathloc,stage
0,prg1_dk0,AB,16.818130,mid,POINT (21.794 138.72),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (38.595 139.485),2.058651,0.008604,PMT
160,prg1_dk10,AB,13.355502,mid,POINT (25.225 139.493),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (38.567 140.101),1.442622,0.006029,PMT
1600,prg1_dk100,AB,22.382311,mid,POINT (16.472 133.278),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (38.831 134.297),7.252778,0.030313,PMT
1616,prg1_dk101,AB,21.825607,mid,POINT (17.102 131.682),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (38.905 132.675),8.875792,0.037096,PMT
1632,prg1_dk102,AB,17.702667,mid,POINT (21.386 128.241),POINT (38.501 141.542),POINT (39.457 120.56),-21.94768,986.54961,0.0,POINT (39.07 129.047),12.508224,0.052277,PMT


In [98]:
final.tail()

,nuc_id,seg_name,distance,intersection,point,head,tail,A,C,pathlen,loc,pathseg,pathloc,stage
33166,prg1_lp746,OP,9.386203,tail,POINT (14.586 67.422),POINT (36.718 37.509),POINT (23.646 69.875),-2.47598,128.422012,182.35187,POINT (23.646 69.875),34.906091,0.908018,LP
33406,prg1_lp761,OP,8.365042,tail,POINT (15.728 67.177),POINT (36.718 37.509),POINT (23.646 69.875),-2.47598,128.422012,182.35187,POINT (23.646 69.875),34.906091,0.908018,LP
33470,prg1_lp765,OP,9.665651,tail,POINT (14.53 66.662),POINT (36.718 37.509),POINT (23.646 69.875),-2.47598,128.422012,182.35187,POINT (23.646 69.875),34.906091,0.908018,LP
33550,prg1_lp770,OP,9.328590,tail,POINT (14.984 66.412),POINT (36.718 37.509),POINT (23.646 69.875),-2.47598,128.422012,182.35187,POINT (23.646 69.875),34.906091,0.908018,LP
33726,prg1_lp781,OP,12.763566,tail,POINT (11.345 66.47),POINT (36.718 37.509),POINT (23.646 69.875),-2.47598,128.422012,182.35187,POINT (23.646 69.875),34.906091,0.908018,LP


### Optional: Select a Random Subset to Display

Plotting the complete set of nuclei and their lines is too dense to make any sense. Execute this code cell to get a random sample for the plot.  

If you might want to reproduce the same sample later pass an optional start state parameter to `coords.sample`.

In [ ]:
# plot_coords = intersections.sample(len(intersections))    # <- choose all points
plot_coords = intersections.sample(250, random_state=0)
# plot_coords = intersections[intersections['point'].y > 100].sample(50)
# plot_coords = intersections[(intersections['point'].y < 20) & (intersections['point'].x < 60)].sample(100)

# see cell below that plots a histogram of distances; use this line to display points based on a distance cutoff
# plot_coords = intersections[intersections['distance'] < 20]
# plot_coords = intersections[intersections['distance'] < 12]

### Plot

Augment the plot shown earlier to include lines from nuclei to their closest line segment. 

In [ ]:
p = figure(title='demo', x_axis_label='x', y_axis_label='y', match_aspect=True)

yellow = Sunset10[5]
px = list(plot_coords['point'].x)
py = list(plot_coords['point'].y)
p.scatter(x=px, y=py, size=8, fill_color=yellow)

for i, row in plot_coords.iterrows():
    xs = [row.point.x, row['loc'].x]
    ys = [row.point.y, row['loc'].y]
    p.line(xs, ys, line_dash='dashed')

xs = list(measurements['point'].x)
ys = list(measurements['point'].y)
ts = list(measurements['name'])

p.line(x=xs, y=ys, color='black')
p.scatter(x=xs, y=ys, size=5, color='black')
p.text(x=xs, y=ys, text=ts, x_offset=5, y_offset=-5)

show(p)

In [ ]:
len(px)

In [ ]:
intersections['distance'].plot(kind='hist', bins=50)

In [ ]:
p = figure(title='demo', x_axis_label='x', y_axis_label='y', match_aspect=True)

df = pd.DataFrame({
    'x': plot_coords['point'].x,
    'y': plot_coords['point'].y,
    'c': plot_coords['pathloc'],
})

cmap = linear_cmap('c', palette="Viridis256", low=0, high=1)

r = p.scatter(x='x', y='y', color=cmap, size=8, source=df)

xs = list(measurements['point'].x)
ys = list(measurements['point'].y)
ts = list(measurements['name'])

p.line(x=xs, y=ys, color='black')
p.scatter(x=xs, y=ys, size=3, color='black')
p.text(x=xs, y=ys, text=ts, x_offset=5, y_offset=-5)

color_bar = r.construct_color_bar(padding=0, ticker=p.xaxis.ticker, formatter=p.xaxis.formatter)
p.add_layout(color_bar, 'below')

show(p)